<a href="https://colab.research.google.com/github/vidyawadhai13/OS-project/blob/main/TAE_1(OS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import copy
import sys

# Constants (same as C)
MAX_DEPT = 5
MAX_RES = 2  # printers, scanners

# Global variables (same as C)
Total = [0] * MAX_RES
Alloc = [[0] * MAX_RES for _ in range(MAX_DEPT)]
Max = [[0] * MAX_RES for _ in range(MAX_DEPT)]
Need = [[0] * MAX_RES for _ in range(MAX_DEPT)]
Avail = [0] * MAX_RES
finished = [False] * MAX_DEPT
safeSeq = [0] * MAX_DEPT
nDept = 0

def computeNeed():
    global Need
    for i in range(nDept):
        for j in range(MAX_RES):
            Need[i][j] = Max[i][j] - Alloc[i][j]

def computeAvail():
    global Avail
    sumAlloc = [0] * MAX_RES
    for i in range(nDept):
        for j in range(MAX_RES):
            sumAlloc[j] += Alloc[i][j]
    for j in range(MAX_RES):
        Avail[j] = Total[j] - sumAlloc[j]

def isSafe():
    global finished, safeSeq
    work = Avail[:]
    for i in range(nDept):
        finished[i] = False

    count = 0
    while count < nDept:
        found = False
        for i in range(nDept):
            if not finished[i]:
                canProceed = True
                for j in range(MAX_RES):
                    if Need[i][j] > work[j]:
                        canProceed = False
                        break
                if canProceed:
                    for j in range(MAX_RES):
                        work[j] += Alloc[i][j]
                    safeSeq[count] = i
                    count += 1
                    finished[i] = True
                    found = True
                    break
        if not found:
            print("System is NOT in safe state.")
            return False

    print("System is in safe state.")
    print("Safe sequence: ", end="")
    for i in range(nDept):
        print(f"D{safeSeq[i]}", end=" ")
    print()
    return True

def requestResource(dept, reqPrint, reqScan):
    req = [reqPrint, reqScan]

    # Check if request exceeds Need
    for j in range(MAX_RES):
        if req[j] > Need[dept][j]:
            print("Request denied: requested more than declared max.")
            return False

    # Check if enough Available
    for j in range(MAX_RES):
        if req[j] > Avail[j]:
            print("Request denied: not enough available resources.")
            return False

    # Save current state
    savedAlloc = copy.deepcopy(Alloc)
    savedAvail = Avail[:]

    # Tentative allocation
    for j in range(MAX_RES):
        Alloc[dept][j] += req[j]
        Avail[j] -= req[j]
    computeNeed()

    # Check if safe
    if isSafe():
        print(f"Request GRANTED to department D{dept}.")
        return True
    else:
        # Restore old state
        for i in range(nDept):
            for j in range(MAX_RES):
                Alloc[i][j] = savedAlloc[i][j]
        for j in range(MAX_RES):
            Avail[j] = savedAvail[j]
        computeNeed()
        print("Request DENIED: granting this may lead to unsafe state.")
        return False

def main():
    global nDept

    # 1. Read basic info (EXACT same prompts as C)
    print(f"Enter number of departments (max {MAX_DEPT}): ", end="")
    nDept = int(input())

    print("Enter total printers and scanners: ", end="")
    Total[0], Total[1] = map(int, input().split())

    # 2. Read allocation matrix
    print("Enter allocation matrix (for each department: printers scanners):")
    for i in range(nDept):
        print(f"D{i}: ", end="")
        Alloc[i][0], Alloc[i][1] = map(int, input().split())

    # 3. Read max matrix
    print("Enter max matrix (max printers and scanners for each dept):")
    for i in range(nDept):
        print(f"D{i}: ", end="")
        Max[i][0], Max[i][1] = map(int, input().split())

    computeNeed()
    computeAvail()

    print("\nInitial state:")
    print(f"Total printers = {Total[0]}, scanners = {Total[1]}")
    print(f"Available: printers={Avail[0]}, scanners={Avail[1]}")

    isSafe()

    # 4. Read one request from user
    print(f"\nEnter request: department (0..{nDept-1}) printers scanners: ", end="")
    dept, reqPrint, reqScan = map(int, input().split())

    if dept < 0 or dept >= nDept:
        print("Invalid department number.")
    else:
        requestResource(dept, reqPrint, reqScan)

if __name__ == "__main__":
    main()


Enter number of departments (max 5): 4
Enter total printers and scanners: 3 2
Enter allocation matrix (for each department: printers scanners):
D0: 1 0 
D1: 1 1
D2: 0 1
D3: 1 0
Enter max matrix (max printers and scanners for each dept):
D0: 2 1
D1: 2 2
D2: 1 2
D3: 2 1

Initial state:
Total printers = 3, scanners = 2
Available: printers=0, scanners=0
System is NOT in safe state.

Enter request: department (0..3) printers scanners: 1 1 1
Request denied: not enough available resources.
